# Tareas

## 1. Diseña una base de datos Cassandra para dar servicio a las lecturas y escrituras anteriores. Argumenta tus decisiones de diseño.

En Cassandra no se diseña “por entidades y relaciones”, sino por consultas. El enunciado da 3 lecturas y 2 escrituras, así que el diseño va a salir directamente de ahí.

Lo que vamos a hacer es es desnormalizar y crear tablas específicas para cada patrón de acceso, en vez de intentar reproducir el modelo relacional.

### Tablas que diseñaríamos:

#### Tabla del perfil del usuario

Hemos decidido incluir una tabla conceptual llamada `user_by_email` con el objetivo de poder resolver de forma rápida la información básica de cada jugador a partir de su email. Esta decisión la hemos tomado porque, al analizar las operaciones del sistema, vimos que las escrituras llegan utilizando el email como dato principal, mientras que algunas lecturas necesitan recuperar campos como el `user_id`. Por tanto, nos interesa que esa conversión entre email e identidad del jugador sea directa y eficiente.

La clave primaria de esta tabla entonces será `email`, que actuará como **partition key**. 
Además, la tabla contendrá las columnas `email`, `user_id`, `user_name` y `country`. Hemos escogido exactamente estos campos porque son los que necesitamos reutilizar después en distintas consultas del sistema. Por ejemplo, en el **Hall of Fame** necesitamos mostrar el `user_name`, en el **Top Horde** necesitamos `user_id`, `user_name` y `email` y además, sabemos que los rankings son locales por país, lo que también supone qu necesitamos obtener el `country` de forma inmediata.

Aunque esta tabla no representa un leaderboard como tal, en el diseño de Cassandra nos parece fundamental porque evita tener que hacer joins entre tablas. En lugar de depender de relaciones complejas en tiempo de consulta, preferimos desnormalizar la información y dejar preparado un acceso directo a los datos más importantes del usuario.

#### Tabla de catálogo de mazmorras

Hemos decidido incluir una tabla conceptual llamada `dungeon_by_id` con el objetivo de poder traducir de forma directa cada `dungeon_id` a su correspondiente `dungeon_name`. Esta decisión la hemos tomado porque, al revisar las consultas que debe soportar el sistema, vimos que no bastaba con almacenar únicamente el identificador de la mazmorra, sino que también necesitábamos poder  disponer de su nombre de forma inmediata, para poder devolver la información completa que exige la aplicación.

La clave primaria de esta tabla será `dungeon_id`, que actuará como identificador único de cada mazmorra. Además, la tabla contendrá las columnas `dungeon_id` y `dungeon_name`. Hemos escogido esta estructura porque es suficiente para resolver exactamente la necesidad que tenemos, sin añadir información innecesaria y manteniendo un acceso muy simple y eficiente.

Aunque esta tabla tampoco representa un leaderboard en sí, nos parece necesaria dentro del diseño general porque permite recuperar el nombre de la mazmorra sin depender de operaciones adicionales. En concreto, en el **Hall of Fame** se devuelve tanto el `dungeon_id` como el `dungeon_name`, por lo que necesitamos que esa traducción pueda hacerse de forma directa y rápida.

#### Tabla para estadísticas de usuario por mazmorra

Hemos decidido incluir una tabla conceptual llamada `runs_by_user_dungeon` para responder a la consulta que recibe como entrada `user_id` y `dungeon_id` y debe devolver una lista de partidas ordenada por menor tiempo, con los campos `{time_minutes, date}`.

La clave primaria estará formada por `((user_id, dungeon_id), time_minutes, date)`. Es decir, usaremos `user_id` y `dungeon_id` como **partition key** compuesta, y `time_minutes` y `date` como **clustering keys**, ambas en orden ascendente. De esta forma, cada partición agrupa las runs de un usuario en una mazmorra concreta, y dentro de ella los resultados quedan ya ordenados por menor tiempo.

Hemos elegido este diseño porque la consulta siempre fija `user_id` y `dungeon_id`, y además necesita que la salida aparezca ordenada por tiempo. Así, Cassandra puede devolver directamente los resultados en el orden que pide el leaderboard, sin necesidad de hacer ordenaciones adicionales en lectura.

#### Tabla para Hall of Fame por país y mazmorra

Hemos decidido utilizar la tabla conceptual `hall_of_fame_by_country_dungeon` para cubrir la lectura del **Hall of Fame**. Esta consulta recibe como entrada un `country` y debe devolver, para cada mazmorra del juego, el **Top 5** de jugadores más rápidos de ese país, incluyendo `email`, `user_name`, `time_minutes` y `date`.

La clave primaria será `((country, dungeon_id), time_minutes, date, email)`. Es decir, usaremos `country` y `dungeon_id` como **partition key** compuesta, y `time_minutes`, `date` y `email` como **clustering keys** en orden ascendente. Además, la tabla contendrá las columnas `country`, `dungeon_id`, `dungeon_name`, `email`, `user_name`, `time_minutes` y `date`.

Hemos elegido este diseño porque permite una lectura muy eficiente por mazmorra. Para servir la consulta, el servidor obtiene primero la lista de mazmorras y después consulta cada partición `(country, dungeon_id)` con `LIMIT 5`. Con esas respuestas construye el JSON final del leaderboard.

Nos parece la mejor opción porque la escritura sigue siendo sencilla, ya que al terminar una dungeon basta con insertar una nueva fila. Además, evitamos mantener listas materializadas o estructuras agregadas más difíciles de actualizar. Aunque esta lectura no sale de una sola query de Cassandra, sino de varias, una por mazmorra, lo consideramos asumible porque el número de mazmorras será finito y controlado y porque este ranking no es una parte crítica del gameplay.

También valoramos como alternativa guardar directamente el Top 5 materializado, pero no la hemos escogido como diseño principal porque obligaría a recalcular y mantener ese top en cada escritura, complicando bastante más la lógica de actualización.

#### Tabla de contador de muertes por jugador en una horda

Hemos decidido incluir una tabla conceptual llamada `horde_kills_by_country_event_user` para almacenar el acumulado de kills de cada jugador dentro de una horda concreta. Esta tabla parte de que la escritura llega con `event_id`, `email` y `monster_id`, mientras que la lectura del **Top Horde** necesita obtener, para un `country`, un `event_id` y un valor `K`, los jugadores con más kills.

La clave primaria será `((country, event_id), user_id)`. Es decir, usaremos `country` y `event_id` como **partition key** compuesta, y `user_id` como **clustering key**. Además, la tabla contendrá las columnas `country`, `event_id`, `user_id`, `email`, `user_name` y `n_killed`.

Hemos elegido este diseño porque cada evento de tipo “user kills monster” debe actualizar el contador de kills del jugador dentro de esa horda concreta, y esta tabla nos permite guardar ese acumulado de forma directa.

Aun así, esta tabla por sí sola no resuelve la lectura del ranking, ya que aunque `n_killed` se actualice correctamente como columna, Cassandra no permite obtener de forma eficiente el top K ordenado por ese valor si no forma parte de la clave. Por eso, hemos concluido que esta tabla es necesaria para acumular la información, pero debe complementarse con otra tabla orientada específicamente al ranking.

#### Tabla de ranking de Horde ordenada por número de kills

Hemos decidido incluir una tabla conceptual llamada `top_horde_by_country_event` para responder de forma eficiente a la lectura del **Top Horde**. Esta consulta recibe como entrada `country`, `event_id` y `K`, y debe devolver los `K` jugadores con mayor número de kills en esa horda.

La clave primaria será `((country, event_id), n_killed, user_id)`. Es decir, usaremos `country` y `event_id` como **partition key** compuesta, y `n_killed` y `user_id` como **clustering keys**. El orden de clustering será `n_killed DESC` y `user_id ASC`. Además, la tabla contendrá las columnas `country`, `event_id`, `user_id`, `user_name`, `email` y `n_killed`.

Hemos elegido este diseño porque encaja de forma muy natural con la lectura que pide el sistema. Al consultar la partición `(country, event_id)` y aplicar `LIMIT K`, Cassandra ya devuelve directamente los jugadores ordenados de mayor a menor número de kills, que es exactamente lo que necesita el leaderboard.